# Ejercicios 1 y 2 - Ambiente y sistema de descarga

Este notebook corre dentro del contenedor `lab8-lab`. La carpeta `data/` del
proyecto esta montada en `/workspace/data`.

## Ejercicio 1 - Preparacion del ambiente

### 1.1 a 1.3. Fork, clonacion y servicios

```bash
git clone <url-del-fork>
cd duckdb
docker compose up --build -d
docker compose ps        # lab8-lab y lab8-metabase deben estar "Up"
```

- JupyterLab: <http://localhost:8888>
- Metabase: <http://localhost:3000>

La siguiente celda verifica desde el propio ambiente que Python y las bibliotecas
principales estan disponibles.

In [1]:
import lab_utils  # fija la raiz del proyecto como directorio de trabajo
import sys, duckdb, pandas, pyarrow, matplotlib, requests

print("Python    ", sys.version.split()[0])
for modulo in (duckdb, pandas, pyarrow, matplotlib, requests):
    print(f"{modulo.__name__:10}", modulo.__version__)

Python     3.11.14
duckdb     1.5.5
pandas     3.0.6
pyarrow    25.0.1
matplotlib 3.11.2
requests   2.34.2


### 1.4. Herramientas disponibles

- **Python 3.11**: lenguaje de automatizacion y analisis.
- **DuckDB**: motor analitico para consultar directamente archivos Parquet.
- **JupyterLab**: ambiente interactivo para notebooks (este).
- **Pandas y PyArrow**: manipulacion de datos e interoperabilidad con Parquet.
- **Matplotlib**: visualizacion.
- **Requests**: descarga automatizada de archivos.
- **Metabase**: construccion de indicadores y tableros (servicio `lab8-metabase`).

### 1.6. Importancia de un ambiente reproducible

Los resultados dependen de los datos y del software que los procesa. Con Docker y
versiones fijadas en `requirements.txt` todos los integrantes y revisores ejecutan
el mismo Python, las mismas bibliotecas y la misma version del driver de DuckDB en
Metabase. Asi se evita el "en mi maquina si funciona", el ambiente se levanta con un
solo comando, queda versionado en Git junto al codigo y no interfiere con otros
proyectos. El detalle esta en la seccion 1.6 del `README.md`.

## Ejercicio 2 - Sistema de descarga

### 2.1 y 2.6. Analisis y cambios al script

El script recibido (`scripts/download_data.py`) ya recorria los doce meses,
permitia elegir Yellow o Green, evitaba repetir archivos y usaba un archivo temporal.
Su debilidad era considerar valido cualquier archivo con tamanio mayor que cero, lo que
podia aceptar descargas interrumpidas. Los cambios fueron:

- `es_parquet_valido` comprueba la firma `PAR1` al inicio y al final del archivo;
- la condicion para omitir un archivo existente usa esa validacion y no solo el tamanio;
- el archivo temporal se valida antes de renombrarlo como `.parquet`;
- un archivo local invalido se informa y se vuelve a descargar.

### 2.2 a 2.5. Ejecucion

Los archivos se guardan en `data/raw/<tipo>/<anio>/`. Al ejecutarlo de nuevo, los
archivos validos aparecen como "ya existian" (requisito 2.4).

In [2]:
!python scripts/download_data.py --years 2026


=== YELLOW 2026 ===
  2026-01  ya existe, se omite


  2026-02  ya existe, se omite
  2026-03  ya existe, se omite
  2026-04  ya existe, se omite
  2026-05  ya existe, se omite


  2026-06  ya existe, se omite
  2026-07  ya existe, se omite
  2026-08  ya existe, se omite


  2026-09  aun no publicado por la TLC


  2026-10  aun no publicado por la TLC


...

  2026-06  ya existe, se omite
  2026-07  ya existe, se omite
  2026-08  ya existe, se omite


  2026-09  aun no publicado por la TLC


  2026-10  aun no publicado por la TLC


  2026-11  aun no publicado por la TLC


  2026-12  aun no publicado por la TLC

Tabla de zonas ya existe: data/raw/zones/taxi_zone_lookup.csv

RESUMEN
  descargados   : 0
  ya existian   : 16
  no publicados : 8
      yellow 2026-09, yellow 2026-10, yellow 2026-11, yellow 2026-12, green 2026-09, green 2026-10, green 2026-11, green 2026-12
  fallidos      : 0


### 2.7. Verificacion de completitud

Contamos con DuckDB los archivos locales por tipo y anio y los comparamos con los
meses publicados por la TLC. Los meses que el servidor reporta como no publicados no
se consideran faltantes.

In [3]:
con = lab_utils.conectar()
con.execute("""
    SELECT regexp_extract(file, '(yellow|green)', 1) AS tipo_taxi,
           regexp_extract(file, 'tripdata_(\\d{4})-', 1) AS anio,
           count(*) AS archivos,
           min(regexp_extract(file, '-(\\d{2})\\.parquet$', 1)) AS primer_mes,
           max(regexp_extract(file, '-(\\d{2})\\.parquet$', 1)) AS ultimo_mes
    FROM glob('data/raw/*/*/*.parquet')
    GROUP BY ALL ORDER BY anio, tipo_taxi
""").df()

,tipo_taxi,anio,archivos,primer_mes,ultimo_mes
0,green,2024,12,01,12
1,yellow,2024,12,01,12
2,green,2025,12,01,12
3,yellow,2025,12,01,12
4,green,2026,8,01,08
5,yellow,2026,8,01,08


Ademas, DuckDB pudo abrir todos los archivos y encontro registros en cada uno
(ver notebook `03_exploracion_parquet`). Las tres comprobaciones (conteo contra lo
publicado, segunda ejecucion sin descargas y lectura con DuckDB) permiten concluir que
la descarga esta completa respecto de lo publicado.